# The Weyl Cycle Clock: Age and Fate of the Universe**Paper III - Study 06**## ObjectiveCalculate the complete timeline of the Weyl Curvature Cycle in the Evaporating Universe:- When did Weyl peak?- What percentage of the cycle is complete?- When will the universe "die" (DoF → 1, Weyl → 0)?### TerminologyIn the Evaporating Universe framework:- **Cosmic Spark** (not "Big Bang"): The dissipative phase transition that initiated our aeon- **Big Freeze**: The asymptotic state where only radiation remains---

In [ ]:
import numpy as npimport matplotlib.pyplot as pltfrom scipy.optimize import brentq, minimize_scalarfrom scipy.integrate import quadimport osimport json# Create output directoriesos.makedirs('figures', exist_ok=True)os.makedirs('results', exist_ok=True)plt.style.use('seaborn-v0_8-whitegrid')plt.rcParams['figure.figsize'] = (12, 8)plt.rcParams['font.size'] = 12print("="*60)print("THE WEYL CYCLE CLOCK")print("Age and Fate of the Evaporating Universe")print("="*60)print("\nTerminology: 'Cosmic Spark' replaces 'Big Bang'")print("(Dissipative phase transition, not singularity)")

## 1. Model Parameters

In [ ]:
# Evaporating Universe Parametersclass EUParams:H0 = 73.2  # km/s/MpcOmega_m0 = 0.30Omega_r0 = 9.0e-5Omega_phi0 = 1 - Omega_m0 - Omega_r0w0 = -1.2z_trans = 0.22# Age of universe today (since Cosmic Spark)t_today = 13.8  # Gyrparams = EUParams()print(f"Parameters: H₀={params.H0}, Ωm={params.Omega_m0}, w₀={params.w0}")print(f"Universe age today (since Cosmic Spark): {params.t_today} Gyr")

## 2. Effective Degrees of Freedom Model

In [ ]:
def cosmic_time_to_z(t, t0=13.8):"""Approximate conversion from cosmic time (since Cosmic Spark) to redshift."""if t <= 0:return np.infif t >= t0:return 0return (t0 / t)**(2/3) - 1def z_to_cosmic_time(z, t0=13.8):"""Approximate conversion from redshift to cosmic time."""if z < 0:return t0 * (1 - z * 0.5)return t0 / (1 + z)**(3/2)def density_fractions(t, params=params):"""Calculate density fractions at cosmic time t."""z = cosmic_time_to_z(t, params.t_today)if z == np.inf:return 0, 1, 0  # Pure radiation at Cosmic Sparkrho_m = params.Omega_m0 * (1 + z)**3rho_r = params.Omega_r0 * (1 + z)**4rho_phi = params.Omega_phi0total = rho_m + rho_r + rho_phireturn rho_m/total, rho_r/total, rho_phi/totaldef effective_dof(t, params=params):"""Effective degrees of freedom at cosmic time t.DoF = 6*f_m + 1*f_r (simplified)"""f_m, f_r, f_phi = density_fractions(t, params)return 6 * f_m + 1 * f_rprint("\nDoF at key epochs (since Cosmic Spark):")for t in [0.001, 1, 5, 10, 13.8]:print(f"  t = {t:5.1f} Gyr: DoF = {effective_dof(t):.2f}")

## 3. The Weyl Cycle ModelWe model the DoF evolution as:- **Phase 1 (Cosmic Spark → Peak)**: DoF rises from 1 to 6- **Phase 2 (Peak → Big Freeze)**: DoF falls from 6 to 1 (evaporation)

In [ ]:
def dof_full_cycle(t, t_peak=5.0, dof_min=1.0, dof_max=6.0, tau_rise=3.0, tau_fall=None):"""Full Weyl Cycle DoF evolution.Parameters:-----------t : floatCosmic time in Gyr (since Cosmic Spark)t_peak : floatTime of maximum DoF (peak structure formation)dof_min : floatMinimum DoF (radiation: 1)dof_max : floatMaximum DoF (matter: 6)tau_rise : floatTimescale for DoF to rise (structure formation)tau_fall : floatTimescale for DoF to fall (evaporation)"""if t <= 0:return dof_minif t < t_peak:# Rising phase: structure formationreturn dof_min + (dof_max - dof_min) * (1 - np.exp(-t / tau_rise))else:# Falling phase: evaporationif tau_fall is None:tau_fall = 4.8return dof_min + (dof_max - dof_min) * np.exp(-(t - t_peak) / tau_fall)# Fit tau_fall to match DoF_todaydof_today_observed = 1.8  # From phase space calculationt_today = 13.8t_peak = 5.0def fit_tau(tau):return abs(dof_full_cycle(t_today, t_peak=t_peak, tau_fall=tau) - dof_today_observed)result = minimize_scalar(fit_tau, bounds=(1, 100), method='bounded')tau_evap_fitted = result.xprint(f"\nFitted evaporation timescale: τ_evap = {tau_evap_fitted:.2f} Gyr")print(f"Verification: DoF(t=13.8 Gyr) = {dof_full_cycle(t_today, t_peak=t_peak, tau_fall=tau_evap_fitted):.2f}")

## 4. Calculate Key Milestones

In [ ]:
# Key milestonesdof_thresholds = {"99% evaporated": 1.05,"95% evaporated": 1.25,"90% evaporated": 1.50,"Today": dof_today_observed,"50% evaporated": 3.5,}def find_time_for_dof(target_dof, t_peak=5.0, tau_fall=tau_evap_fitted):"""Find the cosmic time when DoF reaches target value (in falling phase)."""if target_dof >= 6:return t_peakif target_dof <= 1:return np.inft = t_peak - tau_fall * np.log((target_dof - 1) / 5)return tprint("\n" + "="*60)print("WEYL CYCLE MILESTONES (since Cosmic Spark)")print("="*60)print()milestones = {}for name, dof_target in dof_thresholds.items():t = find_time_for_dof(dof_target)years_from_now = t - t_todayprogress = (6 - dof_target) / (6 - 1) * 100milestones[name] = {"dof": dof_target,"cosmic_time_Gyr": round(t, 2),"years_from_now_Gyr": round(years_from_now, 2),"progress_percent": round(progress, 1)}print(f"{name:20s}: t = {t:6.1f} Gyr | DoF = {dof_target:.2f} | {progress:5.1f}% complete")if years_from_now > 0:print(f"{'':20s}  (in {years_from_now:.1f} Gyr from now)")elif years_from_now < 0:print(f"{'':20s}  ({-years_from_now:.1f} Gyr ago)")print()

## 5. The Weyl Cycle Clock Visualization

In [ ]:
# Create the Weyl Cycle Clockfig = plt.figure(figsize=(14, 10))# Main timeline plotax1 = fig.add_subplot(2, 2, (1, 2))# Time array from Cosmic Spark to far futuret_arr = np.linspace(0.01, 50, 1000)dof_arr = [dof_full_cycle(t, t_peak=t_peak, tau_fall=tau_evap_fitted) for t in t_arr]# Plot DoF evolutionax1.fill_between(t_arr, 1, dof_arr, alpha=0.3, color='blue', label='Active Phase Space')ax1.plot(t_arr, dof_arr, 'b-', lw=3, label='Effective DoF')# Mark key eventsax1.axvline(t_today, color='green', ls='--', lw=2, label=f'Today ({t_today} Gyr)')ax1.axvline(t_peak, color='orange', ls=':', lw=2, label=f'Peak ({t_peak} Gyr)')ax1.scatter([t_today], [dof_today_observed], s=200, c='green', zorder=5, marker='*')# Mark death threshold (99% evaporated)t_death = find_time_for_dof(1.05)ax1.axvline(t_death, color='red', ls='--', lw=2, alpha=0.7, label=f'"Death" (~{t_death:.0f} Gyr)')# Annotations - COSMIC SPARK instead of Big Bangax1.annotate('Cosmic Spark\nWeyl ≈ 0', xy=(0.5, 1.5), fontsize=11,bbox=dict(boxstyle='round', facecolor='gold', alpha=0.8))ax1.annotate('Structure\nPeak', xy=(t_peak, 5.5), fontsize=11, ha='center',bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))ax1.annotate(f'WE ARE\nHERE', xy=(t_today, dof_today_observed + 0.5), fontsize=12,ha='center', fontweight='bold', color='green')ax1.annotate('Big Freeze\nWeyl → 0', xy=(t_death - 5, 1.5), fontsize=11,bbox=dict(boxstyle='round', facecolor='lightcoral', alpha=0.8))ax1.set_xlabel('Cosmic Time since Cosmic Spark (Gyr)', fontsize=14)ax1.set_ylabel('Effective Degrees of Freedom', fontsize=14)ax1.set_title('The Weyl Cycle: From Cosmic Spark to Big Freeze', fontsize=16, fontweight='bold')ax1.legend(loc='upper right', fontsize=10)ax1.set_xlim(0, 50)ax1.set_ylim(0.5, 7)ax1.grid(True, alpha=0.3)# Clock visualization (pie chart style)ax2 = fig.add_subplot(2, 2, 3, projection='polar')# Progress through the cycleprogress_today = (6 - dof_today_observed) / (6 - 1)  # 0 to 1theta_today = progress_today * 2 * np.pi# Draw the clocktheta = np.linspace(0, 2*np.pi, 100)ax2.fill_between(theta, 0, 1, alpha=0.2, color='gray')ax2.fill_between(np.linspace(0, theta_today, 50), 0, 1, alpha=0.6, color='blue', label=f'{progress_today*100:.1f}% Complete')# Mark positions - COSMIC SPARKax2.scatter([0], [0.8], s=100, c='gold', label='Cosmic Spark', edgecolors='orange', linewidths=2)ax2.scatter([np.pi], [0.8], s=100, c='yellow', label='Structure Peak')ax2.scatter([theta_today], [0.8], s=200, c='green', marker='*', label='Today')ax2.scatter([2*np.pi * 0.99], [0.8], s=100, c='red', label='Big Freeze')ax2.set_title('Weyl Cycle Clock', fontsize=14, fontweight='bold', pad=20)ax2.set_ylim(0, 1)ax2.set_yticklabels([])ax2.legend(loc='lower left', bbox_to_anchor=(-0.3, -0.1), fontsize=9)# Summary statisticsax3 = fig.add_subplot(2, 2, 4)ax3.axis('off')summary_text = f"""╔══════════════════════════════════════════╗║       THE WEYL CYCLE SUMMARY             ║╠══════════════════════════════════════════╣║                                          ║║  ✨ Cosmic Spark:       0 Gyr            ║║  🏔️  Structure Peak:  {t_peak:.0f} Gyr   ║║  📍 Today:            {t_today:.1f} Gyr  ║║  💀 Big Freeze (99%): {t_death:.0f} Gyr  ║║                                          ║║  ─────────────────────────────────────   ║║                                          ║║  DoF Today:           {dof_today_observed:.1f}    ║║  Progress:            {progress_today*100:.1f}%   ║║  τ (evaporation):     {tau_evap_fitted:.1f} Gyr   ║║  Time until freeze:   {t_death - t_today:.0f} Gyr ║║                                          ║╚══════════════════════════════════════════╝"""ax3.text(0.5, 0.5, summary_text, transform=ax3.transAxes, fontsize=11,verticalalignment='center', horizontalalignment='center',fontfamily='monospace', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))plt.tight_layout()plt.savefig('figures/weyl_cycle_clock.png', dpi=150, bbox_inches='tight')plt.show()print("\n✅ Figure saved: figures/weyl_cycle_clock.png")

## 6. Results Summary

In [ ]:
results = {"study": "Weyl Cycle Clock","paper": "Paper III","status": "Complete","terminology": {"origin": "Cosmic Spark","note": "Replaces 'Big Bang' - dissipative phase transition, not singularity"},"parameters": {"t_peak_Gyr": t_peak,"t_today_Gyr": t_today,"tau_evap_Gyr": round(tau_evap_fitted, 2),"dof_today": dof_today_observed,"dof_min": 1,"dof_max": 6},"key_results": {"progress_percent": round((6 - dof_today_observed) / 5 * 100, 1),"t_freeze_99_percent_Gyr": round(find_time_for_dof(1.05), 1),"time_until_freeze_Gyr": round(find_time_for_dof(1.05) - t_today, 1),"total_cycle_duration_Gyr": round(find_time_for_dof(1.05), 1)},"milestones": milestones,"interpretation": {"meaning": "The universe is 84% through its complexity cycle","current_state": "Active evaporation phase","fate": "Conformal equivalence to Cosmic Spark (Penrose CCC)"},"figures": ["figures/weyl_cycle_clock.png"]}with open('results/weyl_cycle_results.json', 'w') as f:json.dump(results, f, indent=2)print("="*60)print("WEYL CYCLE COMPLETE")print("="*60)print(json.dumps(results["key_results"], indent=2))print("\n✅ Results saved to: results/weyl_cycle_results.json")

## 7. Download (Colab)

In [ ]:
try:from google.colab import filesfiles.download('figures/weyl_cycle_clock.png')files.download('results/weyl_cycle_results.json')print("\n✅ All files downloaded!")except ImportError:print("Files saved locally:")print("  - figures/weyl_cycle_clock.png")print("  - results/weyl_cycle_results.json")